# Federated Oncology Trial — Training Demo

This notebook demonstrates a complete federated learning workflow for
oncology clinical trials. Two simulated hospital sites collaboratively
train a tumor-response classifier **without exchanging raw patient data**.

## What you will learn

1. Generate synthetic oncology data and partition it across sites.
2. Run Federated Averaging (FedAvg) with a central coordinator.
3. Compare federated vs. centralized training accuracy.
4. Apply differential privacy and observe its impact.
5. Integrate physical AI digital-twin features.

In [ ]:
import sys

sys.path.insert(0, "..")

import numpy as np

from federated.client import FederatedClient
from federated.coordinator import FederationCoordinator
from federated.data_ingestion import (
    DataPartitioner,
    generate_synthetic_oncology_data,
)
from federated.model import FederatedModel, ModelConfig
from physical_ai.digital_twin import PatientDigitalTwin, TumorModel

print("Imports successful!")

## 1. Generate Synthetic Oncology Data

We generate 600 synthetic patient records with 30 clinical features
(tumor measurements, biomarkers, demographics) and a binary label
(favorable vs. poor treatment response).

In [ ]:
X, y = generate_synthetic_oncology_data(n_samples=600, n_features=30, n_classes=2, seed=42)
print(f"Dataset shape: {X.shape}")
print(f"Class distribution: {np.bincount(y)}")

## 2. Partition Data Across Two Hospital Sites

Each hospital keeps its data locally. We use IID partitioning
(each site gets a representative sample) with an 80/20 train/test split.

In [ ]:
partitioner = DataPartitioner(num_sites=2, strategy="iid", test_fraction=0.2, seed=42)
sites = partitioner.partition(X, y)

for site in sites:
    print(f"{site.site_id}: {site.num_train} train, {site.num_test} test")

## 3. Federated Training (FedAvg)

The coordinator distributes a global model to both sites.
Each site trains locally for a few epochs, then sends back
**only model parameters** (never raw data). The coordinator
averages them into a new global model.

In [ ]:
config = ModelConfig(input_dim=30, hidden_dims=[64, 32], output_dim=2, seed=42)
num_rounds = 15

# Initialize coordinator
coordinator = FederationCoordinator(model_config=config, num_rounds=num_rounds)
global_params = coordinator.initialize()

# Create clients
clients = []
for site in sites:
    client = FederatedClient(site.site_id, config)
    client.set_data(site.x_train, site.y_train)
    clients.append(client)

# Track metrics
federated_accuracies = []
federated_losses = []

for r in range(num_rounds):
    updates = []
    counts = []
    for client in clients:
        client.train_local(global_params, epochs=5, lr=0.01)
        updates.append(client.get_parameters())
        counts.append(client.get_sample_count())

    eval_data = (sites[0].x_test, sites[0].y_test)
    result = coordinator.run_round(updates, client_sample_counts=counts, eval_data=eval_data)
    global_params = coordinator.get_global_parameters()

    federated_accuracies.append(result.global_metrics["accuracy"])
    federated_losses.append(result.global_metrics["loss"])

print(f"Final federated accuracy: {federated_accuracies[-1]:.4f}")
print(f"Final federated loss: {federated_losses[-1]:.4f}")

## 4. Centralized Baseline

For comparison, train the same model architecture on all data
combined (as if a single hospital had everything). This is the
upper bound that federated training should approach.

In [ ]:
# Combine all training data
all_x_train = np.vstack([s.x_train for s in sites])
all_y_train = np.concatenate([s.y_train for s in sites])
y_onehot = np.zeros((len(all_y_train), 2))
y_onehot[np.arange(len(all_y_train)), all_y_train] = 1.0

centralized_model = FederatedModel(input_dim=30, hidden_dims=[64, 32], output_dim=2, seed=42)
centralized_accuracies = []

for epoch in range(num_rounds * 5):  # equivalent total epochs
    centralized_model.train_step(all_x_train, y_onehot, lr=0.01)
    if (epoch + 1) % 5 == 0:
        metrics = centralized_model.evaluate(sites[0].x_test, sites[0].y_test)
        centralized_accuracies.append(metrics["accuracy"])

print(f"Final centralized accuracy: {centralized_accuracies[-1]:.4f}")
print(f"Final federated accuracy:   {federated_accuracies[-1]:.4f}")
print(f"Gap: {centralized_accuracies[-1] - federated_accuracies[-1]:.4f}")

## 5. Federated Training with Differential Privacy

Now add differential privacy (epsilon=2.0) to see its impact.
DP adds noise to aggregated model updates, providing a formal
guarantee that no individual patient's data can be inferred.

In [ ]:
dp_coordinator = FederationCoordinator(
    model_config=config,
    num_rounds=num_rounds,
    use_differential_privacy=True,
    dp_epsilon=2.0,
    dp_delta=1e-5,
)
dp_params = dp_coordinator.initialize()

dp_clients = []
for site in sites:
    c = FederatedClient(site.site_id, config)
    c.set_data(site.x_train, site.y_train)
    dp_clients.append(c)

dp_accuracies = []
for r in range(num_rounds):
    updates = []
    counts = []
    for c in dp_clients:
        c.train_local(dp_params, epochs=5, lr=0.01)
        updates.append(c.get_parameters())
        counts.append(c.get_sample_count())

    result = dp_coordinator.run_round(
        updates,
        client_sample_counts=counts,
        eval_data=(sites[0].x_test, sites[0].y_test),
    )
    dp_params = dp_coordinator.get_global_parameters()
    dp_accuracies.append(result.global_metrics["accuracy"])

privacy = dp_coordinator.dp.get_privacy_spent()
print(f"DP accuracy: {dp_accuracies[-1]:.4f}")
print(f"Privacy spent: epsilon={privacy['total_epsilon_spent']:.2f}")
print(f"Accuracy cost of DP: {federated_accuracies[-1] - dp_accuracies[-1]:.4f}")

## 6. Physical AI — Digital Twin Integration

Each hospital creates patient digital twins from clinical data.
The digital twins generate feature vectors for federated training
and can simulate treatment responses locally.

In [ ]:
rng = np.random.default_rng(42)

# Create digital twins for 5 example patients
for i in range(5):
    tumor = TumorModel(
        volume_cm3=float(rng.uniform(0.5, 8.0)),
        growth_rate=float(rng.uniform(30, 120)),
        chemo_sensitivity=float(rng.uniform(0.2, 0.8)),
        radio_sensitivity=float(rng.uniform(0.2, 0.8)),
        location="lung",
    )
    twin = PatientDigitalTwin(
        f"patient_{i:03d}",
        tumor=tumor,
        age=int(rng.integers(40, 75)),
        biomarkers={"pdl1": float(rng.uniform(0, 1)), "ki67": float(rng.uniform(0, 1))},
    )

    # Simulate chemo
    chemo = twin.simulate_treatment("chemotherapy", dose_mg=75, cycles=4)
    # Simulate radiation
    rad = twin.simulate_treatment("radiation", dose_gy=60, fractions=30)

    print(f"Patient {twin.patient_id} | Age: {twin.age} | Tumor: {tumor.volume_cm3:.1f} cm3")
    print(f"  Chemo: {chemo['response_category']} ({chemo['volume_reduction_pct']:.1f}% reduction)")
    print(f"  Radiation: {rad['response_category']} ({rad['volume_reduction_pct']:.1f}% reduction)")
    print(f"  Feature vector shape: {twin.generate_feature_vector().shape}")
    print()

## 7. Summary

| Metric | Centralized | Federated | Federated + DP |
|--------|------------|-----------|----------------|
| Data sharing | All data pooled | No raw data shared | No raw data shared |
| Privacy | None | Aggregation only | Formal DP guarantee |
| Accuracy | Baseline | Close to baseline | Slightly lower |

**Key takeaway:** Federated learning achieves accuracy comparable to
centralized training while keeping all patient data at its originating
hospital. Adding differential privacy provides formal privacy guarantees
with a modest accuracy trade-off.